# Automated LLM-Based Cognitive Ontology Annotation

In functional neuroimaging literature, assigning broad cognitive labels to data-driven clusters can inadvertently introduce researcher bias. To strictly ensure objectivity in our cognitive ontology mapping, this notebook employs a state-of-the-art Large Language Model (LLM) to blindly evaluate and annotate the underlying functional domains of our term clusters.

In [1]:
import httpx
import json
import joblib
from pathlib import Path

DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')
PLOTS_PATH = Path('../plots')
DOCUMENTS_PATH = Path('../documents')

# Data Anonymization

To prevent any semantic leakage or prompt-based confirmation bias, we deliberately strip out our predefined theoretical category labels (e.g., `affect`, `valuation`, `social`). We replace them with completely neutral, arbitrary identifiers (`Cluster 1`, `Cluster 2`, `Cluster 3`). The model will evaluate the latent macroscopic domains based solely on the raw cognitive terms provided.

In [3]:
http_client = httpx.Client(proxy='http://127.0.0.1:7899', timeout=None)

with open(DATA_PATH / 'term_mapping.json', 'r', encoding='utf8') as f:
    term_mapping_dict = json.load(f)
    term_mapping_dict_50 = {k: v for (k, v) in term_mapping_dict.items() if v != "others"}
    term_list_15 = list(term_mapping_dict.keys())[:15]
    term_mapping_dict_15 = {k: v for (k, v) in term_mapping_dict_50.items() if k in term_list_15}

    # anonymize the dict key!!!
    cluster_mapping_dict_15 = {val: [k for k, v in term_mapping_dict_15.items() if v == val] for i, val
                               in enumerate(list(set(term_mapping_dict_15.values())))}
    cluster_mapping_dict_15['Cluster 1'] = cluster_mapping_dict_15.pop('affect')
    cluster_mapping_dict_15['Cluster 2'] = cluster_mapping_dict_15.pop('valuation')
    cluster_mapping_dict_15['Cluster 3'] = cluster_mapping_dict_15.pop('social')
    print(cluster_mapping_dict_15)


    cluster_mapping_dict_50 = {val: [k for k, v in term_mapping_dict_50.items() if v == val] for i, val
                               in enumerate(list(set(term_mapping_dict_50.values())))}
    cluster_mapping_dict_50['Cluster 1'] = cluster_mapping_dict_50.pop('affect')
    cluster_mapping_dict_50['Cluster 2'] = cluster_mapping_dict_50.pop('valuation')
    cluster_mapping_dict_50['Cluster 3'] = cluster_mapping_dict_50.pop('social')
    print(cluster_mapping_dict_50)

{'Cluster 1': ['emotion', 'affective', 'fear', 'depression'], 'Cluster 2': ['reward', 'value', 'decision making', 'food', 'choice', 'motivation'], 'Cluster 3': ['autobiographical', 'social', 'self', 'person', 'theory mind']}
{'Cluster 1': ['emotion', 'affective', 'fear', 'depression', 'autonomic', 'arousal', 'emotion regulation', 'threat', 'anxiety', 'conditioning', 'major depressive', 'happy', 'pain', 'feelings', 'mood', 'aversive', 'stress'], 'Cluster 2': ['reward', 'value', 'decision making', 'food', 'choice', 'motivation', 'reinforcement', 'incentive', 'monetary', 'impulsive', 'drug', 'loss', 'gambling'], 'Cluster 3': ['autobiographical', 'social', 'self', 'person', 'theory mind', 'empathy', 'autism', 'social interactions', 'faces', 'self referential']}


# System Prompt Engineering

We construct a highly constrained system prompt instructing the LLM to act as an expert cognitive neuroscientist. The instructions explicitly forbid the use of anatomical network names (e.g., "Default Mode", "Limbic") or micro-level psychological states. Instead, the model is strictly constrained to synthesize the shared neurobiological co-activation patterns of the terms and output a single, broad functional domain name in a parsable JSON format.

In [4]:
system_prompt_str = '''
Act as an expert cognitive neuroscientist specializing in large-scale brain networks and cognitive ontology.
I will provide you with pre-defined clusters of terms. Your task is to identify the latent macroscopic functional domain for each cluster and assign it a name.

CRITICAL CONSTRAINTS:
1. Analytical Logic: Evaluate the terms in each cluster based on their shared macroscopic neuroimaging co-activation patterns (e.g., Default Mode Network, Reward/Striatal Circuitry, Salience/Limbic Network), rather than general English semantics.
2. Naming Convention (The "What"): The `domain_name` MUST be a macro, broad, high-level *functional or cognitive* category.
   - HARD RULE 1: Do NOT name the domain after anatomical brain networks themselves (e.g., absolutely do NOT output "Salience", "DefaultMode", "Limbic", or "Striatal" as the domain name).
   - HARD RULE 2: Do NOT use specific micro-cognitive processes, lower-level task processes, or specific psychological states (such as "mentalizing", "introspection", "reflection", "simulation", "fear").
3. Formatting: The domain name must be strictly ONE single word.

You MUST output ONLY a JSON object matching this schema:
{
  "clusters": [
    {
      "cluster_id": "Name of the input cluster (e.g., Cluster 1)",
      "domain_name": "FunctionalNameHere",
      "justification": "One sentence explaining the underlying latent cognitive or functional axis that binds these terms together."
    }
  ]
}
'''

user_prompt_str_15 = f"Following are input clusters:\n{cluster_mapping_dict_15}"
user_prompt_str_50 = f"Following are input clusters:\n{cluster_mapping_dict_50}"

# Iterative Querying and Consensus Building

Because generative language models possess inherent variability due to temperature and sampling randomness, a single zero-shot query is insufficient for rigorous scientific validation.

We programmatically query the API for 100 independent iterations for both the 15-term and 50-term sets. By aggregating these 100 independent zero-shot judgments in downstream analyses, we can quantify algorithmic stability and establish a statistically robust consensus for naming the VMPFC functional subregions.

In [5]:
def label_term(system_prompt, user_prompt, model):
    API_KEY = 'sk-or-v1-....'

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "reasoning": {
            "enabled": True,
            "effort": "xhigh",

        }
    }

    response = http_client.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=payload)

    if response.status_code == 200:
        response_data = response.json()
        return response, response_data['choices'][0]['message']['content']
    else:
        raise ValueError(f"API Error: {response.status_code} - {response.text}")

In [6]:
(DATA_PATH / 'responses_15_gemini').parent.mkdir(parents=True, exist_ok=True)
for i in range(1, 101):
    resp_filepath = DATA_PATH / f'responses_15_gemini/resp{i:02}.joblib'
    if resp_filepath.exists():
        print(f"Task {i} skipped (already exists).")
        continue

    try:
        resp, content = label_term(
            system_prompt=system_prompt_str,
            user_prompt=user_prompt_str_15,
            model='google/gemini-3.1-pro-preview'
        )

        joblib.dump(resp, resp_filepath)

        clean_content = content.strip().removeprefix('```json').removesuffix('```').strip()
        resp_json = json.loads(clean_content)

        json_filepath = resp_filepath.with_suffix('.json')
        with open(json_filepath, 'w', encoding='utf8') as f:
            json.dump(resp_json, f, ensure_ascii=False, indent=4)
    except Exception as e:
        print(e)
        continue


Task 1 skipped (already exists).
Task 2 skipped (already exists).
Task 3 skipped (already exists).
Task 4 skipped (already exists).
Task 5 skipped (already exists).
Task 6 skipped (already exists).
Task 7 skipped (already exists).
Task 8 skipped (already exists).
Task 9 skipped (already exists).
Task 10 skipped (already exists).
Task 11 skipped (already exists).
Task 12 skipped (already exists).
Task 13 skipped (already exists).
Task 14 skipped (already exists).
Task 15 skipped (already exists).
Task 16 skipped (already exists).
Task 17 skipped (already exists).
Task 18 skipped (already exists).
Task 19 skipped (already exists).
Task 20 skipped (already exists).
Task 21 skipped (already exists).
Task 22 skipped (already exists).
Task 23 skipped (already exists).
Task 24 skipped (already exists).
Task 25 skipped (already exists).
Task 26 skipped (already exists).
Task 27 skipped (already exists).
Task 28 skipped (already exists).
Task 29 skipped (already exists).
Task 30 skipped (alread

In [7]:
(DATA_PATH / 'responses_50_gemini').parent.mkdir(parents=True, exist_ok=True)
for i in range(1, 101):
    resp_filepath = DATA_PATH / f'responses_50_gemini/resp{i:02}.joblib'
    if resp_filepath.exists():
        print(f"Task {i} skipped (already exists).")
        continue

    try:
        resp, content = label_term(
            system_prompt=system_prompt_str,
            user_prompt=user_prompt_str_50,
            model='google/gemini-3.1-pro-preview'
        )

        joblib.dump(resp, resp_filepath)

        clean_content = content.strip().removeprefix('```json').removesuffix('```').strip()
        resp_json = json.loads(clean_content)

        json_filepath = resp_filepath.with_suffix('.json')
        with open(json_filepath, 'w', encoding='utf8') as f:
            json.dump(resp_json, f, ensure_ascii=False, indent=4)
    except Exception as e:
        print(e)
        continue

Task 1 skipped (already exists).
Task 2 skipped (already exists).
Task 3 skipped (already exists).
Task 4 skipped (already exists).
Task 5 skipped (already exists).
Task 6 skipped (already exists).
Task 7 skipped (already exists).
Task 8 skipped (already exists).
Task 9 skipped (already exists).
Task 10 skipped (already exists).
Task 11 skipped (already exists).
Task 12 skipped (already exists).
Task 13 skipped (already exists).
Task 14 skipped (already exists).
Task 15 skipped (already exists).
Task 16 skipped (already exists).
Task 17 skipped (already exists).
Task 18 skipped (already exists).
Task 19 skipped (already exists).
Task 20 skipped (already exists).
Task 21 skipped (already exists).
Task 22 skipped (already exists).
Task 23 skipped (already exists).
Task 24 skipped (already exists).
Task 25 skipped (already exists).
Task 26 skipped (already exists).
Task 27 skipped (already exists).
Task 28 skipped (already exists).
Task 29 skipped (already exists).
Task 30 skipped (alread